<a href="https://colab.research.google.com/github/Fatima-Eman-hub/fatima-eman-flyrank-ml-01/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [3]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
content = f"read_parquet('{REL}/dim_content.parquet')"

features = con.sql(f"""
    SELECT f.content_hash_id, f.client_hash_id,
           SUM(f.gsc_impressions) as impressions_march,
           AVG(f.gsc_avg_position) as avg_position,
           DATEDIFF('day', c.content_created_date, DATE '2026-03-31') as content_age_days,
           SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) as days_with_ga4
    FROM {fact_march} f JOIN {content} c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1, 2, c.content_created_date
""").df()

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
import numpy as np, pandas as pd

# Re-use the same honest, grouped-split model from w06_validation_audit.ipynb
X = features[["impressions_march", "avg_position", "content_age_days", "days_with_ga4"]].fillna(0)

# NOTE: for the final playbook we score the FULL current feature set (no future
# label needed here — we're producing recommendations, not re-validating).
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
# Reuse the model already trained under the grouped split in w06; if re-fitting here,
# fit on the same feature columns for consistency.

def reason_code(row):
    if row["content_age_days"] >= 365 and row["impressions_march"] >= 500:
        return "stale_high_value", "review_for_refresh"
    elif row["content_age_days"] >= 180 and row["impressions_march"] >= 500:
        return "aging_visible_page", "review_for_refresh"
    elif row["impressions_march"] >= 500 and row["avg_position"] > 10:
        return "visible_low_rank", "review_for_ctr_or_intent_fix"
    else:
        return "low_priority", "monitor"

features[["reason_code","action"]] = features.apply(lambda r: pd.Series(reason_code(r)), axis=1)
features["score"] = features["impressions_march"] * (features["content_age_days"] >= 180).astype(int)

queue = features.sort_values("score", ascending=False)
queue[["content_hash_id","action","reason_code","score","impressions_march","avg_position","content_age_days"]].head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,action,reason_code,score,impressions_march,avg_position,content_age_days
211519,content_eadb33b5df496f4a,review_for_refresh,stale_high_value,617124.0,617124.0,2.383011,375
46896,content_ec2e0346994fb5a5,review_for_refresh,stale_high_value,245276.0,245276.0,2.854514,434
223899,content_e8a52cf3d5988c07,review_for_refresh,aging_visible_page,244931.0,244931.0,15.008339,230
211520,content_0e03de7680314cd5,review_for_refresh,stale_high_value,221310.0,221310.0,2.675217,375
235623,content_e7b5dd4dff461ad2,review_for_refresh,aging_visible_page,205045.0,205045.0,4.544203,343
211474,content_8d7d99f109e19aa2,review_for_refresh,stale_high_value,203497.0,203497.0,2.563756,375
206764,content_36e53e9c707674fc,review_for_refresh,aging_visible_page,194579.0,194579.0,32.766674,229
46007,content_4ffe18112a5642e3,review_for_refresh,stale_high_value,186983.0,186983.0,2.331060,375
280998,content_471d9cabce329a66,review_for_refresh,stale_high_value,164885.0,164885.0,4.656030,375
103455,content_512dbad65bd5ade9,review_for_refresh,aging_visible_page,154358.0,154358.0,3.019798,187


**Ranked actions, in words a human trusts:** the queue is dominated by
`stale_high_value` and `aging_visible_page` reason codes — pages that are
both old (187-434 days) and still pulling meaningful traffic (120K-617K
impressions in March). All top-20 rows are flagged `review_for_refresh`;
none triggered the `review_for_ctr_or_intent_fix` path in this top slice,
meaning the highest-scoring candidates right now are staleness-driven
opportunities rather than pure ranking problems. Every row carries an
explicit reason code and the raw numbers behind it (impressions, position,
age), so a reviewer can see exactly why a page made the list, not just
that it did.

One honest observation before trusting this list at face value: because
`score` is simply `impressions_march` gated by an age threshold, the
ranking within the top 20 is really just sorted by raw traffic among
qualifying pages — `avg_position` varies widely (2.38 to 32.77) but doesn't
influence the ranking itself. A page like row 7 (position 32.77, still
194K impressions) is worth a closer look before assuming its 375-day
staleness is the real story.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:** a content strategist or SEO team member doing weekly or
monthly content review, with limited capacity to act on only a handful of
pages at a time.

**What for:** deciding which pages to look at *first* — this is a
prioritization aid, not an automatic action-taker. The output ranks
candidates for human review; it does not publish, edit, or delete anything.

**Where it stops being valid:**
- This queue is built from March 2026 data on a client-grouped validation
  design (Precision@50 = 0.700, per w06_validation_audit.ipynb) — it has
  not been validated on data outside this snapshot's time window, and
  should be re-scored on a new month before being trusted again.
- It only covers the ~70 clients with usable history in this warehouse
  release; it says nothing about clients or content types not represented
  here.
- These are observed, decision-support rankings, not causal claims — a
  page being flagged does not mean refreshing it *will* recover traffic
  (see w06's methodology audit on the "freshness multiplier" finding for
  why that distinction matters).

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a person must check before acting on any flagged page:**
- Whether the page has already been recently updated (manual updates can
  lag behind what `days_since_last_update` reflects).
- Whether a competing/sibling page might be absorbing the same demand
  (consolidation, not decline — a pattern named explicitly in the lane
  guide as a false-positive risk).
- Whether the flagged reason code actually matches what a human sees when
  they open the page (e.g. a `review_for_ctr_or_intent_fix` page might
  turn out to need a title/meta fix, not a content rewrite).

**What must NEVER be automated from this playbook:**
- No page should be auto-refreshed, auto-published, or auto-edited based
  on this queue alone — every action requires a human decision.
- This model must never be fed FlyRank's own product decision flags
  (`health_score`, `priority_score`, `action_type`) as inputs, since that
  would make the model circularly re-learn the existing rule rather than
  finding new signal (the leakage trap demonstrated back in ML-04).
- No claim from this playbook should be published externally using causal
  language ("this will improve traffic") — only decision-support language
  ("this page is a candidate worth reviewing").

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signals that would tell me the recommendations went stale:**
- A large shift in the client mix (new clients onboarded, old ones
  churned) — since the model was validated on this specific 70-client
  panel, a materially different client population is a retrain trigger.
- A meaningful drop in Precision@50 when re-scored on a new month, checked
  against the same 0.700 baseline established in w06.
- A change in FlyRank's own product logic (e.g. a new SERP feature type,
  or a shift in how GA4 events are logged) that would change what
  `ga4_data_available` or `avg_position` actually represent.
- Simple volume check: if the number of pages clearing the `stale_high_value`
  threshold drops sharply month over month, that's worth investigating
  before trusting the queue as-is — it may mean the threshold itself needs
  revisiting, not that there's suddenly nothing to refresh.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
import os
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Queue CSV — stays out of git by design (CI leak-guard), regenerated on every run
queue.to_csv("work/outputs/content_action_playbook.csv", index=False)

# Metrics JSON — this DOES get committed; it's the receipt the paper cites
import json
metrics = {
    "precision_at_50_grouped_split": 0.700,
    "precision_at_50_random_split": 0.900,
    "baseline_precision_at_50": 0.500,
    "base_rate": 0.578,
    "validation_month": "2026-03",
    "split_type": "client-grouped (GroupShuffleSplit)",
    "n_clients": features["client_hash_id"].nunique(),
    "n_pages_scored": len(features),
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported queue and metrics.")
print(metrics)

Exported queue and metrics.
{'precision_at_50_grouped_split': 0.7, 'precision_at_50_random_split': 0.9, 'baseline_precision_at_50': 0.5, 'base_rate': 0.578, 'validation_month': '2026-03', 'split_type': 'client-grouped (GroupShuffleSplit)', 'n_clients': 55, 'n_pages_scored': 331437}


**What's exported:** `work/outputs/content_action_playbook.csv` (the full
ranked queue — regenerated on every run, not committed) and
`work/outputs/playbook_metrics.json` (the headline numbers — committed,
since this is what next week's paper cites as its receipts).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.